# Class Conditioning & Generation
**TabDDPM: Hospital Readmission Prediction**

This notebook covers:
1. Validating that class conditioning is working correctly in the model
2. Running the reverse diffusion loop to generate synthetic readmission patients
3. Generating at 50%, 100%, and 200% augmentation levels
4. Building augmented training sets for the evaluation stage
5. Sanity checking synthetic patients against real ones

---
## Setup & Imports

In [ ]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import sys

# Make sure tabddpm_model.py and generate.py can be imported from this folder
sys.path.append(os.path.dirname(os.path.abspath('')))

from tabddpm_model import TabDDPMDenoiser, validate_class_conditioning
from generate import generate_patients

# File paths — adjust these if your folder structure is different
CONFIG_PATH   = '../models/diffusion_config.pkl'
MODEL_PATH    = '../models/tabddpm.pt'
X_TRAIN_PATH  = '../data/processed/X_train.csv'
Y_TRAIN_PATH  = '../data/processed/y_train.csv'
SYNTHETIC_DIR = '../data/synthetic'

os.makedirs(SYNTHETIC_DIR, exist_ok=True)
print('All imports successful.')

---
## Cell 1 — Load Config and Check Column Counts

Load the diffusion config and confirm the column breakdown is correct.
We expect 9 numeric columns, 33 categorical columns, 42 total.

In [ ]:
with open(CONFIG_PATH, 'rb') as f:
    cfg = pickle.load(f)

print('Config keys:', list(cfg.keys()))
print(f"Numeric columns : {len(cfg['numeric_indices'])}")
print(f"Categorical cols: {len(cfg['cat_indices'])}")
print(f"Total features  : {len(cfg['numeric_indices']) + len(cfg['cat_indices'])}")

---
## Cell 2 — Instantiate the Model

Build the model using the column information from the config.
No trained weights yet — this just sets up the architecture.

In [ ]:
model = TabDDPMDenoiser(
    num_numeric       = len(cfg['numeric_indices']),
    cat_cardinalities = cfg['cat_num_classes'],
    numeric_indices   = cfg['numeric_indices'],
    cat_indices       = cfg['cat_indices'],
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model instantiated. Total parameters: {total_params:,}')

---
## Cell 3 — Validate Class Conditioning

This check confirms that the readmission label (0 or 1) actually flows
through the network and produces a non-zero gradient on the class embedding weights.

This uses dummy data — no trained weights needed.
If the gradient sum is greater than 0, conditioning is working correctly.

In [ ]:
grad_sum = validate_class_conditioning(
    model,
    batch_size     = 32,
    total_features = 42,
)

print(f"Class conditioning gradient sum: {grad_sum:.4f}")
print("PASS — class label is flowing through the network correctly.")

---
## Cell 4 — Load Trained Weights

Once `tabddpm.pt` is available, drop it in the `models/` folder and run this cell.
Everything from Cell 5 onward depends on the trained weights being loaded.

In [ ]:
# Run this once tabddpm.pt is in the models/ folder
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()
print('Trained weights loaded successfully.')

---
## Cell 5 — Generate Synthetic Patients at 3 Augmentation Levels

We generate at 50%, 100%, and 200% of the minority class size.
The minority class has ~9,051 patients in training, so:
- 50%  → ~4,525 synthetic patients
- 100% → ~9,051 synthetic patients
- 200% → ~18,102 synthetic patients

In [ ]:
y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()
minority_count = (y_train == 1).sum()
print(f'Minority class (readmitted <30) in training set: {minority_count:,}')

for pct in [50, 100, 200]:
    n = int(minority_count * pct / 100)
    print(f'\n--- {pct}% augmentation: generating {n:,} synthetic patients ---')

    synthetic = generate_patients(
        model_path   = MODEL_PATH,
        config_path  = CONFIG_PATH,
        X_train_path = X_TRAIN_PATH,
        y_train_path = Y_TRAIN_PATH,
        n_samples    = n,
        target_class = 1,
    )

    out_path = os.path.join(SYNTHETIC_DIR, f'synthetic_{pct}pct.csv')
    synthetic.to_csv(out_path, index=False)
    print(f'Saved -> {out_path}')

---
## Cell 6 — Build Augmented Training Sets

Combines the real training data with the synthetic minority patients.
The output files are what the evaluation stage will use to train the XGBoost classifier.

In [ ]:
X_train = pd.read_csv(X_TRAIN_PATH)
y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()

print(f'Original training set: {len(X_train):,} rows')
print(f'Minority class: {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)')
print()

for pct in [50, 100, 200]:
    synthetic = pd.read_csv(os.path.join(SYNTHETIC_DIR, f'synthetic_{pct}pct.csv'))
    n = len(synthetic)

    # Add synthetic patients to the real training set
    X_aug = pd.concat([X_train, synthetic], ignore_index=True)
    y_aug = pd.concat([y_train, pd.Series([1] * n)], ignore_index=True)

    X_aug.to_csv(os.path.join(SYNTHETIC_DIR, f'X_train_aug_{pct}pct.csv'), index=False)
    y_aug.to_csv(os.path.join(SYNTHETIC_DIR, f'y_train_aug_{pct}pct.csv'), index=False)

    total_minority = (y_train == 1).sum() + n
    print(f'{pct}%: {len(X_aug):,} total rows | '
          f'{total_minority:,} minority ({total_minority/len(X_aug)*100:.1f}%)')

print('\nAugmented training sets saved to data/synthetic/')

---
## Cell 7 — Sanity Check: One Synthetic Patient vs. Real Ones

Prints a side-by-side comparison of one synthetic patient against the mean
of real readmitted patients. The values should be in a similar range — not
identical, but plausible.

In [ ]:
synthetic_50 = pd.read_csv(os.path.join(SYNTHETIC_DIR, 'synthetic_50pct.csv'))
X_train      = pd.read_csv(X_TRAIN_PATH)
y_train      = pd.read_csv(Y_TRAIN_PATH).squeeze()
real_minority = X_train[y_train == 1]

comparison = pd.DataFrame({
    'synthetic_patient'     : synthetic_50.iloc[0].round(3),
    'real_mean (readmitted)': real_minority.mean().round(3),
    'real_std (readmitted)' : real_minority.std().round(3),
})

print('=== Synthetic patient vs. real readmitted patients ===\n')
print(comparison.to_string())

---
## Cell 8 — Distribution Plots (Real vs. Synthetic)

Side-by-side histograms for key numeric features.
If the distributions look similar, the generated patients are realistic.
This plot is saved to results/ for use in the final report.

In [ ]:
import matplotlib.pyplot as plt

synthetic_50  = pd.read_csv(os.path.join(SYNTHETIC_DIR, 'synthetic_50pct.csv'))
X_train       = pd.read_csv(X_TRAIN_PATH)
y_train       = pd.read_csv(Y_TRAIN_PATH).squeeze()
real_minority = X_train[y_train == 1]

# The 4 most interpretable numeric columns
PLOT_COLS = ['time_in_hospital', 'num_medications', 'num_lab_procedures', 'number_inpatient']

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('Real vs. Synthetic Readmission Patients — Feature Distributions', fontsize=13)

for i, col in enumerate(PLOT_COLS):
    if col not in real_minority.columns:
        continue

    axes[0, i].hist(real_minority[col].dropna(), bins=30, color='steelblue',
                    edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col}\n(real)', fontsize=10)
    axes[0, i].set_ylabel('Count' if i == 0 else '')

    axes[1, i].hist(synthetic_50[col].dropna(), bins=30, color='tomato',
                    edgecolor='white', alpha=0.85)
    axes[1, i].set_title(f'{col}\n(synthetic)', fontsize=10)
    axes[1, i].set_ylabel('Count' if i == 0 else '')

plt.tight_layout()
plt.savefig('../results/real_vs_synthetic_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/real_vs_synthetic_distributions.png')